[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/24_ensemble_learning/03_ensemble_design.ipynb)

# Bài 03 — Thiết kế ensemble

**🎯 Mục tiêu**
- Các nguồn tạo **đa dạng**: dữ liệu, feature, thuật toán, siêu tham số, seed.
- **Ensemble selection** (Caruana et al., 2004): chọn tham lam từ thư viện model — cài đặt from scratch; nguy cơ overfit validation.
- Đánh đổi **độ chính xác ↔ chi phí** (thời gian dự đoán, bảo trì); distillation.
- Bất đồng giữa thành viên như **tín hiệu độ bất định**.
- Template model cho ensemble.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/24_ensemble_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from ml_course.visualization import set_style

set_style()


def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

## 1. Nguồn đa dạng

| Nguồn | Cách làm | Ví dụ |
|---|---|---|
| Dữ liệu | mẫu con, bootstrap, trọng số mẫu | bagging, boosting |
| Feature | tập con feature, biểu diễn khác nhau | random subspace, Random Forest, feature engineering khác nhau |
| Thuật toán | họ model khác nhau | tuyến tính + láng giềng + kernel + cây |
| Siêu tham số | cùng thuật toán, cấu hình khác | Ridge với nhiều α, cây nông/sâu |
| Ngẫu nhiên | seed khởi tạo, thứ tự dữ liệu | deep ensembles, snapshot ensembles |
| Mục tiêu | loss/target khác | quantile, log-target, phân loại thứ hạng |

Đa dạng **thuật toán** thường hiệu quả nhất cho dữ liệu bảng; đa dạng **seed** hiệu quả cho mạng nơ-ron (chương 37).

## 2. Thư viện model

Chia dữ liệu giá nhà thành **train / validation / test**. Train 19 cấu hình thuộc 6 họ model:

In [ ]:
house = pd.read_csv("../../data/sample/house_prices.csv")
y = house.pop("price_billion_vnd").to_numpy()
num_cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]


def prep():
    return ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"]),
                              ("bin", "passthrough", ["has_garden"])], sparse_threshold=0)


X_tmp, X_te, y_tmp, y_te = train_test_split(house, y, test_size=0.3, random_state=0)
X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=0.3, random_state=1)

library = {}
for a in [0.1, 1, 10, 100]:
    library[f"Ridge α={a}"] = make_pipeline(prep(), RidgeCV(alphas=[a]))
for k in [5, 15, 40]:
    library[f"KNN k={k}"] = make_pipeline(prep(), KNeighborsRegressor(k))
for C in [1, 10, 100]:
    library[f"SVR C={C}"] = make_pipeline(prep(), SVR(C=C))
for d in [4, 8, None]:
    library[f"Tree depth={d}"] = make_pipeline(prep(), DecisionTreeRegressor(max_depth=d, random_state=0))
for mf in [0.3, 1.0]:
    library[f"RF max_features={mf}"] = make_pipeline(prep(), RandomForestRegressor(300, max_features=mf, random_state=0, n_jobs=-1))
library["ExtraTrees"] = make_pipeline(prep(), ExtraTreesRegressor(300, random_state=0, n_jobs=-1))
for lr in [0.03, 0.1, 0.3]:
    library[f"HistGB lr={lr}"] = make_pipeline(prep(), HistGradientBoostingRegressor(learning_rate=lr, random_state=0))

P_val, P_te, pred_ms = {}, {}, {}
for name, model in library.items():
    model.fit(X_tr, y_tr)
    P_val[name] = model.predict(X_val)
    t0 = time.perf_counter()
    P_te[name] = model.predict(X_te)
    pred_ms[name] = (time.perf_counter() - t0) * 1000
names = list(library)
V = np.column_stack([P_val[n] for n in names])
T = np.column_stack([P_te[n] for n in names])
table = pd.DataFrame({"RMSE val": [rmse(y_val, V[:, j]) for j in range(len(names))],
                      "RMSE test": [rmse(y_te, T[:, j]) for j in range(len(names))],
                      "dự đoán (ms)": [pred_ms[n] for n in names]}, index=names).sort_values("RMSE val")
table.round(4)

## 3. Ensemble selection (Caruana et al., 2004)

Thuật toán tham lam trên **validation**:
1. Bắt đầu với ensemble rỗng.
2. Mỗi bước, thêm (**có hoàn lại**) model làm RMSE validation của **trung bình** ensemble giảm nhiều nhất.
3. Lặp một số bước cố định; trọng số = số lần mỗi model được chọn.

Chọn có hoàn lại = cho phép trọng số khác nhau; đơn giản, khó overfit hơn tối ưu trọng số tự do.

In [ ]:
def ensemble_selection(V, y_val, n_steps=30):
    counts = np.zeros(V.shape[1])
    current = np.zeros(len(y_val))
    history = []
    for step in range(n_steps):
        candidate_scores = [rmse(y_val, (current * step + V[:, j]) / (step + 1)) for j in range(V.shape[1])]
        j = int(np.argmin(candidate_scores))
        counts[j] += 1
        current = (current * step + V[:, j]) / (step + 1)
        history.append(candidate_scores[j])
    return counts / counts.sum(), np.array(history)


w, hist = ensemble_selection(V, y_val)
best_single = table.index[0]
rows = {"model đơn tốt nhất (theo val)": (table.loc[best_single, "RMSE val"], table.loc[best_single, "RMSE test"]),
        "trung bình đều 19 model": (rmse(y_val, V.mean(1)), rmse(y_te, T.mean(1))),
        "ensemble selection": (rmse(y_val, V @ w), rmse(y_te, T @ w))}
print(pd.DataFrame(rows, index=["RMSE val", "RMSE test"]).T.round(4))
print("\ntrọng số được chọn:", {names[j]: round(w[j], 3) for j in np.flatnonzero(w)})

In [ ]:
plt.plot(np.arange(1, len(hist) + 1), hist, marker=".")
plt.xlabel("số bước chọn")
plt.ylabel("RMSE validation của ensemble")
plt.title("Giảm nhanh trong vài bước đầu, sau đó dao động — dừng sớm hoặc cố định số bước")
plt.show()

Ensemble selection chọn một nhóm nhỏ model **khác họ** (tuyến tính + kernel + boosting) và vượt rõ cả model đơn tốt nhất lẫn trung bình đều. Lưu ý:
- RMSE **validation** của ensemble được tối ưu trực tiếp → lạc quan; luôn báo cáo trên **test** riêng (như trên). Validation nhỏ → chọn nhiễu; Caruana đề xuất **bagging quá trình chọn** (chọn trên nhiều mẫu con của thư viện) và khởi đầu với vài model tốt nhất.
- Model đơn tốt nhất theo validation chưa chắc tốt nhất trên test — chọn **một** model cũng là một dạng overfit validation.

## 4. Độ chính xác ↔ chi phí

In [ ]:
sel = [names[j] for j in np.flatnonzero(w)]
cost_rows = [("model đơn tốt nhất", rmse(y_te, T[:, names.index(best_single)]), pred_ms[best_single], 1),
             ("ensemble selection", rmse(y_te, T @ w), sum(pred_ms[n] for n in sel), len(sel)),
             ("trung bình 19 model", rmse(y_te, T.mean(1)), sum(pred_ms.values()), len(names))]
cost = pd.DataFrame(cost_rows, columns=["cấu hình", "RMSE test", f"thời gian dự đoán {len(y_te)} mẫu (ms)", "số model phải chạy"]).set_index("cấu hình")
cost.round(3)

Ensemble đắt hơn: chạy nhiều model, nhiều pipeline phải bảo trì, giám sát, cập nhật thư viện. Câu hỏi cần trả lời trước khi triển khai: **mức cải thiện có đáng chi phí không** (RMSE giảm bao nhiêu → bao nhiêu tiền)? Các lựa chọn giảm chi phí:
- **Chọn ít thành viên** (ensemble selection tự nhiên cho tập nhỏ); bỏ model chậm có trọng số thấp.
- **Distillation**: train một model đơn (vd HistGB) bắt chước đầu ra của ensemble trên nhiều dữ liệu (kể cả dữ liệu không nhãn) — giữ phần lớn chất lượng với chi phí một model (Bucilă et al., 2006; Hinton et al., 2015).
- Chạy ensemble **offline** (batch) khi không cần độ trễ thấp.

## 5. Bất đồng = độ bất định

Khi các thành viên **bất đồng** mạnh về một căn nhà, dự đoán trung bình có đáng tin không?

In [ ]:
top6 = list(table.index[:6])
T6 = np.column_stack([P_te[n] for n in top6])
spread = T6.std(1)
abs_err = np.abs(y_te - T6.mean(1))
rho = spearmanr(spread, abs_err).statistic
q = pd.qcut(spread, 5, labels=["thấp nhất", "2", "3", "4", "cao nhất"])
by_q = pd.Series(abs_err).groupby(q, observed=True).mean()
print(f"tương quan hạng (độ lệch chuẩn giữa 6 model, |sai số|) = {rho:.3f}")
print("sai số tuyệt đối trung bình theo nhóm độ bất đồng:\n", by_q.round(3).to_string())

Nhóm căn nhà mà các model bất đồng nhiều nhất có sai số lớn gấp nhiều lần nhóm đồng thuận → độ bất đồng là tín hiệu rẻ để **gắn cờ** dự đoán cần người kiểm tra. Đây cũng là ý tưởng của **deep ensembles** để ước lượng độ bất định của mạng nơ-ron (chương 37) — nhưng đó chưa phải khoảng tin cậy được hiệu chỉnh.

## 6. Template model: Ensemble

| Mục | Nội dung |
|---|---|
| Problem | mọi bài toán có giám sát khi cần thêm độ chính xác/ổn định |
| Intuition | nhiều model giỏi, sai khác chỗ nhau → lỗi triệt tiêu |
| Math | phân rã ambiguity; bỏ phiếu đa số; bias–variance |
| Algorithm | voting/averaging, stacking (OOF), ensemble selection, bagging, boosting |
| Complexity | tổng chi phí các thành viên (+ K lần fit cho OOF) |
| Hyperparameters | tập thành viên, trọng số, meta-model, số bước chọn |
| Assumptions | thành viên tốt hơn ngẫu nhiên, lỗi không tương quan hoàn toàn |
| Strengths | thường tốt hơn model đơn, ổn định, tín hiệu bất định |
| Weaknesses | chậm, khó diễn giải, khó bảo trì, dễ rò rỉ khi stacking |
| Failure cases | thành viên yếu/giống nhau; stacking in-sample; chọn trên validation nhỏ |
| Production | cân nhắc chi phí; distillation; giám sát từng thành viên |
| Alternatives | một model được tuning tốt; distillation |

## 🧠 Kiểm tra nhanh

1. Kể 4 nguồn tạo đa dạng.
2. Ensemble selection hoạt động thế nào? Vì sao chọn có hoàn lại?
3. Vì sao RMSE validation của ensemble selection là lạc quan?
4. Khi nào không nên triển khai ensemble?
5. Độ bất đồng giữa thành viên dùng vào việc gì?

## 📌 Tóm tắt

- Đa dạng đến từ dữ liệu, feature, thuật toán, siêu tham số, seed, mục tiêu.
- Ensemble selection tham lam trên validation — đơn giản, mạnh; đánh giá trên test riêng.
- Cân nhắc chi phí; distillation; bất đồng = tín hiệu bất định.

## 📚 Tài liệu tham khảo

- Caruana, Niculescu-Mizil, Crew & Ksikes (2004) — *Ensemble Selection from Libraries of Models*
- Hinton, Vinyals & Dean (2015) — *Distilling the Knowledge in a Neural Network*
- Lakshminarayanan, Pritzel & Blundell (2017) — *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*